# Semester Major Project: Preprocessing, Regularisation and Generalisation in EEG Classification

This notebook investigates how preprocessing choices, model complexity and regularisation strategies affect generalisation performance in seizure prediction and EEG classification tasks.

**Datasets used:**
1. Epileptic Seizure Recognition dataset
2. EEG Signal dataset
3. ADHD EEG dataset used as an additional EEG-based generalisation dataset

**Important note:** The third dataset is ADHD vs Control, not seizure vs non-seizure. It is included as an additional EEG classification dataset to test whether preprocessing and regularisation behaviour generalises across a different EEG task.

## 1. Upload and Extract Dataset ZIP

Upload one ZIP file containing these three CSV files:

- `Epileptic Seizure Recognition.csv`
- `EEG_Signal.csv`
- `adhdata.csv`

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os
from pathlib import Path

zip_files = [name for name in uploaded.keys() if name.lower().endswith('.zip')]

if len(zip_files) == 0:
    raise FileNotFoundError("Please upload one ZIP file containing the three CSV datasets.")

zip_path = zip_files[0]
extract_dir = Path('/content/ML_Project')
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extracted files:")
for path in extract_dir.rglob('*'):
    if path.is_file():
        print(path)

## 2. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, average_precision_score, precision_score, recall_score, PrecisionRecallDisplay
from sklearn.metrics import classification_report, confusion_matrix

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.pipeline import Pipeline as ImbPipeline
except Exception:
    !pip install -q imbalanced-learn
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.pipeline import Pipeline as ImbPipeline

## 3. Dataset Loading and Description

In [ ]:
def find_csv_file(filename):
    matches = list(Path('/content/ML_Project').rglob(filename))
    if len(matches) == 0:
        all_files = [p.name for p in Path('/content/ML_Project').rglob('*.csv')]
        raise FileNotFoundError(f"Could not find {filename}. Available CSV files: {all_files}")
    return str(matches[0])


def describe_dataset(name, X, y):
    print(f"
{name}")
    print("-" * len(name))
    print("Samples:", X.shape[0])
    print("Features:", X.shape[1])
    print("Class distribution:")
    print(pd.Series(y).value_counts().sort_index())
    print("Class ratio:")
    print(pd.Series(y).value_counts(normalize=True).sort_index().round(3))


def extract_1d_signal_features(values):
    values = np.asarray(values, dtype=float)
    fft_vals = np.abs(np.fft.rfft(values))
    return {
        "mean": np.mean(values),
        "std": np.std(values),
        "min": np.min(values),
        "max": np.max(values),
        "median": np.median(values),
        "q25": np.percentile(values, 25),
        "q75": np.percentile(values, 75),
        "energy": np.sum(values ** 2) / len(values),
        "abs_mean": np.mean(np.abs(values)),
        "zero_crossings": np.sum(np.diff(np.sign(values)) != 0),
        "fft_mean": np.mean(fft_vals),
        "fft_std": np.std(fft_vals),
        "fft_max": np.max(fft_vals)
    }


def load_epileptic_recognition():
    path = find_csv_file('Epileptic Seizure Recognition.csv')
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    if "Unnamed" in df.columns:
        df = df.drop(columns=["Unnamed"])
    y = (df["y"] == 1).astype(int)
    X = df.drop(columns=["y"])
    return X, y, "Epileptic Seizure Recognition"


def load_eeg_signal():
    path = find_csv_file('EEG_Signal.csv')
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    rows = []
    for segment_id, group in df.groupby("id person"):
        label = group["Labels"].iloc[0]
        features = extract_1d_signal_features(group["Signal"].values)
        features["target"] = 1 if label == "E" else 0
        rows.append(features)
    features_df = pd.DataFrame(rows)
    y = features_df["target"].astype(int)
    X = features_df.drop(columns=["target"])
    return X, y, "EEG Signal Dataset"


def load_adhd_eeg():
    path = find_csv_file('adhdata.csv')
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    channel_cols = [c for c in df.columns if c not in ["Class", "ID"]]
    agg_dict = {}
    for col in channel_cols:
        agg_dict[col] = ['mean', 'std', 'min', 'max']
    features_df = df.groupby('ID').agg(agg_dict)
    features_df.columns = [f'{col}_{stat}' for col, stat in features_df.columns]
    labels = df.groupby('ID')['Class'].first()
    y = (labels == 'ADHD').astype(int).reset_index(drop=True)
    X = features_df.reset_index(drop=True)
    return X, y, "ADHD EEG Dataset"


datasets = {}
for loader in [load_epileptic_recognition, load_eeg_signal, load_adhd_eeg]:
    X, y, name = loader()
    datasets[name] = (X, y)
    describe_dataset(name, X, y)

## 4. Preprocessing Pipelines

Two different preprocessing pipelines are compared.

**Pipeline A:** Imputation → Scaling → Feature Selection → Logistic Regression

**Pipeline B:** Imputation → Scaling → PCA → Logistic Regression

Pipeline A keeps the strongest individual features, while Pipeline B transforms the feature space using PCA. This allows testing whether preprocessing order and feature transformation affect generalisation.

In [ ]:
def safe_k_features(X):
    return max(1, min(30, X.shape[1]))


def safe_pca_components(X):
    return max(1, min(10, X.shape[1], X.shape[0] - 1))


def make_logistic_model(penalty='l2', C=1.0, class_weight=None, l1_ratio=None):
    if penalty == 'elasticnet':
        return LogisticRegression(
            penalty='elasticnet', solver='saga', l1_ratio=l1_ratio,
            C=C, class_weight=class_weight, max_iter=5000, random_state=42
        )
    if penalty == 'l1':
        return LogisticRegression(
            penalty='l1', solver='saga', C=C,
            class_weight=class_weight, max_iter=5000, random_state=42
        )
    if penalty == 'l2':
        return LogisticRegression(
            penalty='l2', solver='lbfgs', C=C,
            class_weight=class_weight, max_iter=5000, random_state=42
        )
    if penalty == 'none':
        return LogisticRegression(
            penalty=None, solver='lbfgs', class_weight=class_weight,
            max_iter=5000, random_state=42
        )


def build_pipeline(pipeline_name, X, model, sampler=None):
    if pipeline_name == 'Pipeline A':
        steps = [
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('feature_selection', SelectKBest(score_func=f_classif, k=safe_k_features(X)))
        ]
    elif pipeline_name == 'Pipeline B':
        steps = [
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=safe_pca_components(X), random_state=42))
        ]
    else:
        raise ValueError('Unknown pipeline name')

    if sampler is not None:
        return ImbPipeline(steps + [('sampler', sampler), ('model', model)])
    return Pipeline(steps + [('model', model)])

## 5. Evaluation Function

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)
    return {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'PR_AUC': average_precision_score(y_test, y_score)
    }

## 6. Baseline Logistic Regression Results

In [ ]:
baseline_results = []

for dataset_name, (X, y) in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    for pipeline_name in ['Pipeline A', 'Pipeline B']:
        model = make_logistic_model(penalty='l2', C=1.0)
        pipe = build_pipeline(pipeline_name, X_train, model)
        metrics = evaluate_model(pipe, X_train, X_test, y_train, y_test)
        metrics.update({
            'Dataset': dataset_name,
            'Pipeline': pipeline_name,
            'Regularisation': 'Baseline L2',
            'Imbalance_Method': 'None'
        })
        baseline_results.append(metrics)

baseline_df = pd.DataFrame(baseline_results)
baseline_df = baseline_df[['Dataset', 'Pipeline', 'Regularisation', 'Imbalance_Method', 'Accuracy', 'Precision', 'Recall', 'F1', 'PR_AUC']]
baseline_df.round(4)

## 7. Regularisation Study: L1, L2 and Elastic Net

L1 regularisation can create sparse models by forcing some coefficients to zero. L2 regularisation keeps all features but shrinks coefficients smoothly. Elastic Net combines both behaviours.

In [ ]:
regularisation_settings = [
    ('L1', {'penalty': 'l1', 'C': 1.0}),
    ('L2', {'penalty': 'l2', 'C': 1.0}),
    ('Elastic Net', {'penalty': 'elasticnet', 'C': 1.0, 'l1_ratio': 0.5})
]

regularisation_results = []

for dataset_name, (X, y) in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    for pipeline_name in ['Pipeline A', 'Pipeline B']:
        for reg_name, params in regularisation_settings:
            model = make_logistic_model(**params)
            pipe = build_pipeline(pipeline_name, X_train, model)
            metrics = evaluate_model(pipe, X_train, X_test, y_train, y_test)
            fitted_model = pipe.named_steps['model']
            non_zero = np.sum(np.abs(fitted_model.coef_) > 1e-6)
            total_coefficients = fitted_model.coef_.size
            sparsity = 1 - (non_zero / total_coefficients)
            metrics.update({
                'Dataset': dataset_name,
                'Pipeline': pipeline_name,
                'Regularisation': reg_name,
                'Imbalance_Method': 'None',
                'Non_Zero_Coefficients': non_zero,
                'Sparsity': sparsity
            })
            regularisation_results.append(metrics)

regularisation_df = pd.DataFrame(regularisation_results)
regularisation_df = regularisation_df[['Dataset', 'Pipeline', 'Regularisation', 'Imbalance_Method', 'Accuracy', 'Precision', 'Recall', 'F1', 'PR_AUC', 'Non_Zero_Coefficients', 'Sparsity']]
regularisation_df.round(4)

## 8. Class Imbalance Handling

Three imbalance strategies are tested: class weighting, SMOTE oversampling and random undersampling. The impact is evaluated using precision, recall, F1-score and PR-AUC.

In [ ]:
imbalance_settings = [
    ('None', None, None),
    ('Class Weight', None, 'balanced'),
    ('SMOTE', SMOTE(random_state=42, k_neighbors=3), None),
    ('Undersampling', RandomUnderSampler(random_state=42), None)
]

imbalance_results = []

for dataset_name, (X, y) in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    for pipeline_name in ['Pipeline A', 'Pipeline B']:
        for method_name, sampler, class_weight in imbalance_settings:
            model = make_logistic_model(penalty='l2', C=1.0, class_weight=class_weight)
            pipe = build_pipeline(pipeline_name, X_train, model, sampler=sampler)
            metrics = evaluate_model(pipe, X_train, X_test, y_train, y_test)
            metrics.update({
                'Dataset': dataset_name,
                'Pipeline': pipeline_name,
                'Regularisation': 'L2',
                'Imbalance_Method': method_name
            })
            imbalance_results.append(metrics)

imbalance_df = pd.DataFrame(imbalance_results)
imbalance_df = imbalance_df[['Dataset', 'Pipeline', 'Regularisation', 'Imbalance_Method', 'Accuracy', 'Precision', 'Recall', 'F1', 'PR_AUC']]
imbalance_df.round(4)

## 9. Combined Final Results Table

In [ ]:
combined_results = pd.concat([
    baseline_df,
    regularisation_df.drop(columns=['Non_Zero_Coefficients', 'Sparsity'], errors='ignore'),
    imbalance_df
], ignore_index=True)

combined_results = combined_results.drop_duplicates()
combined_results.round(4)

In [ ]:
best_results = combined_results.sort_values(['Dataset', 'PR_AUC', 'F1'], ascending=[True, False, False])
best_results.groupby('Dataset').head(5).round(4)

## 10. Visual Comparison of F1 and PR-AUC

In [ ]:
summary_plot_df = regularisation_df.copy()

for metric in ['F1', 'PR_AUC']:
    plt.figure(figsize=(12, 5))
    labels = summary_plot_df['Dataset'] + ' | ' + summary_plot_df['Pipeline'] + ' | ' + summary_plot_df['Regularisation']
    plt.bar(range(len(summary_plot_df)), summary_plot_df[metric])
    plt.xticks(range(len(summary_plot_df)), labels, rotation=90)
    plt.ylabel(metric)
    plt.title(f'{metric} Comparison Across Datasets, Pipelines and Regularisation')
    plt.tight_layout()
    plt.show()

## 11. Precision-Recall Curves

In [ ]:
for dataset_name, (X, y) in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    plt.figure(figsize=(7, 5))
    for pipeline_name in ['Pipeline A', 'Pipeline B']:
        model = make_logistic_model(penalty='l2', C=1.0)
        pipe = build_pipeline(pipeline_name, X_train, model)
        pipe.fit(X_train, y_train)
        y_score = pipe.predict_proba(X_test)[:, 1]
        PrecisionRecallDisplay.from_predictions(y_test, y_score, name=pipeline_name, ax=plt.gca())
    plt.title(f'Precision-Recall Curves: {dataset_name}')
    plt.tight_layout()
    plt.show()

## 12. Overfitting and Underfitting Demonstration

Underfitting is created using very strong regularisation and limited features. Overfitting is encouraged using weak regularisation and high-dimensional input. Learning curves are used to compare training and validation behaviour.

In [ ]:
def make_underfit_pipeline(X):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('feature_selection', SelectKBest(score_func=f_classif, k=max(1, min(3, X.shape[1])))),
        ('model', make_logistic_model(penalty='l2', C=0.001))
    ])


def make_overfit_pipeline(X):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', make_logistic_model(penalty='l2', C=1000))
    ])


def plot_learning_curve_for_model(model, X, y, title):
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        cv=cv,
        scoring='f1',
        train_sizes=np.linspace(0.2, 1.0, 5),
        n_jobs=-1
    )
    train_mean = np.mean(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    plt.figure(figsize=(7, 5))
    plt.plot(train_sizes, train_mean, marker='o', label='Training F1')
    plt.plot(train_sizes, val_mean, marker='o', label='Validation F1')
    plt.xlabel('Training examples')
    plt.ylabel('F1-score')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

for dataset_name, (X, y) in datasets.items():
    plot_learning_curve_for_model(make_underfit_pipeline(X), X, y, f'Underfitting Learning Curve: {dataset_name}')
    plot_learning_curve_for_model(make_overfit_pipeline(X), X, y, f'Overfitting Learning Curve: {dataset_name}')

## 13. Final Comparative Analysis

### Does preprocessing order affect results?
The results show that preprocessing changes the feature representation before logistic regression learns the decision boundary. Pipeline A keeps selected original features, while Pipeline B creates transformed PCA components. Differences in F1-score and PR-AUC across the two pipelines show that preprocessing order and feature transformation can affect model performance.

### Which regularisation generalises best across datasets?
L2 regularisation is often more stable because it shrinks all coefficients without removing features completely. L1 can be useful when many irrelevant features exist because it creates sparse models. Elastic Net can balance both effects, but it does not automatically outperform L1 or L2 in every dataset.

### Does Elastic Net consistently outperform L1/L2?
Elastic Net does not always consistently outperform L1 or L2. Its performance depends on dataset size, feature correlation and class imbalance. This is why comparing it across all three datasets is important.

### How does imbalance handling interact with regularisation?
Imbalance handling changes the precision-recall trade-off. SMOTE and class weighting may improve recall for the minority class, but this can reduce precision. Regularisation controls coefficient complexity, while imbalance handling changes the effective decision boundary learned from the data. The best setup should therefore be selected using F1-score and PR-AUC, not accuracy alone.

## 14. Export Results for Report

In [ ]:
combined_results.to_csv('final_combined_results.csv', index=False)
regularisation_df.to_csv('regularisation_results.csv', index=False)
imbalance_df.to_csv('imbalance_results.csv', index=False)

print('Saved result files:')
print('final_combined_results.csv')
print('regularisation_results.csv')
print('imbalance_results.csv')